# 06 — Compute Player Form Features

Compute short-term player form from the previous five matches played before each shot,
then save train/test datasets for the form-aware xG model.

- **Notebook 05** captures long-term skill across all prior shots (cumulative career history).
- **Notebook 06** captures short-term form across the last five matches played (rolling window).

**New features added:**
- `goals_last_5_matches` — goals scored in the player's previous 5 matches
- `shots_last_5_matches` — shot attempts in the player's previous 5 matches
- `conversion_rate_last_5_matches` — goals / shots in form window; 0.0 when no history
- `matches_in_form_window` — how many prior matches are in the window (0–5)
- `xg_sum_last_5_matches` — sum of xg_pred across the form window (optional extra)
- `goals_minus_xg_last_5_matches` — goals over expected in form window (optional extra)
- `shots_per_match_last_5_matches` — average shots per match in form window (optional extra)
- `xg_per_shot_last_5_matches` — average xG per shot in form window (optional extra)

In [ ]:
import sys

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from src.config import INTERIM_DIR as DATA_DIR

## Step 0: Path validation

In [ ]:
TRAIN_PATH = DATA_DIR / "wyscout_train_xg_skill.parquet"
TEST_PATH  = DATA_DIR / "wyscout_test_xg_skill.parquet"

assert TRAIN_PATH.exists(), "Run 05_compute_skill_features.ipynb first"
assert TEST_PATH.exists(),  "Run 05_compute_skill_features.ipynb first"
print("Paths OK")

## Step 1: Load skill-enriched datasets

In [ ]:
train = pd.read_parquet(TRAIN_PATH)
test  = pd.read_parquet(TEST_PATH)

n_train = len(train)
n_test  = len(test)

assert n_train > 0 and n_test > 0, "Empty train or test set"

# xg_pred is a model probability (from notebook 04) — verify before using in rolling extras
assert train["xg_pred"].between(0, 1).all(), "xg_pred out of [0, 1] in train"
assert test["xg_pred"].between(0, 1).all(),  "xg_pred out of [0, 1] in test"

REQUIRED_COLS = {
    "league", "matchId", "playerId", "teamId",
    "match_datetime_utc", "match_date",
    "period_sort_key", "seconds_from_match_start", "shot_sequence_in_match",
    "eventSec", "matchPeriod",
    "is_goal", "xg_pred",
    "career_shots_before_shot", "career_goals_before_shot",
    "career_conversion_rate_before_shot", "has_prior_shot_history",
}
assert REQUIRED_COLS.issubset(train.columns), \
    f"Missing train columns: {REQUIRED_COLS - set(train.columns)}"
assert REQUIRED_COLS.issubset(test.columns), \
    f"Missing test columns: {REQUIRED_COLS - set(test.columns)}"

assert set(train["league"].unique()) == {"France", "Germany", "Italy", "Spain"}, \
    "Wrong leagues in train"
assert set(test["league"].unique()) == {"England"}, "Wrong leagues in test"

train = train.copy()
test  = test.copy()
train["dataset_split"] = "train"
test["dataset_split"]  = "test"

summary = pd.DataFrame({
    "split":          ["train", "test"],
    "rows":           [n_train, n_test],
    "goal_rate":      [train["is_goal"].mean(), test["is_goal"].mean()],
    "unique_players": [train["playerId"].nunique(), test["playerId"].nunique()],
})
display(summary)

## Step 2: Combine and sort globally

Form must be computed on the combined dataset sorted globally.
Splitting and computing separately would truncate the form window for players
who appear in both splits (cross-league transfers).

**Cross-league players:** ~20 players appear in both train and test splits.
Computing form globally means a player's England form correctly includes prior
continental-league matches. This is intentional — it is the most chronologically
accurate representation, not a leak.

In [ ]:
SORT_COLS = [
    "match_datetime_utc",
    "matchId",
    "period_sort_key",
    "seconds_from_match_start",
    "shot_sequence_in_match",
    "league",
    "playerId",
    "teamId",
]

shots = pd.concat([train, test], ignore_index=True)

for col in SORT_COLS + ["is_goal", "xg_pred"]:
    assert shots[col].notna().all(), f"Null values in {col} before sort"

shots = shots.sort_values(SORT_COLS, ignore_index=True)

assert shots["match_datetime_utc"].is_monotonic_increasing, \
    "Primary sort key not non-decreasing — sort failed"

ID_COLS = ["league", "matchId", "playerId", "teamId", "eventSec", "matchPeriod"]
assert not shots[ID_COLS].duplicated().any(), "Duplicate shot identities in combined table"

print(f"Combined shots: {len(shots):,}")
print(f"Earliest: {shots['match_datetime_utc'].min()}")
print(f"Latest:   {shots['match_datetime_utc'].max()}")

## Step 3: Build player-match table

Form is a match-level concept: aggregate each player's performance per match first,
then compute rolling windows over those match-level rows.

This ensures all shots by the same player in the same match see identical form features.
Form must not update within a match based on earlier shots in the same match.

In [ ]:
player_match = (
    shots.groupby(["playerId", "matchId"], as_index=False)
    .agg(
        league=("league", "first"),
        match_datetime_utc=("match_datetime_utc", "first"),
        match_date=("match_date", "first"),
        dataset_split=("dataset_split", "first"),
        match_shots=("is_goal", "count"),
        match_goals=("is_goal", "sum"),
        match_xg_sum=("xg_pred", "sum"),
    )
)
# dataset_split="first" is for traceability only.
# Cross-league players appear in both splits, so this column cannot be used analytically.

player_match["match_shots"] = player_match["match_shots"].astype("int32")
player_match["match_goals"] = player_match["match_goals"].astype("int32")

assert not player_match[["playerId", "matchId"]].duplicated().any(), \
    "player_match is not unique per (playerId, matchId)"
assert player_match[["playerId", "matchId", "match_datetime_utc"]].notna().all().all(), \
    "Null key columns in player_match"
assert (player_match["match_goals"] <= player_match["match_shots"]).all(), \
    "match_goals exceeds match_shots"

PLAYER_MATCH_SORT = ["playerId", "match_datetime_utc", "matchId", "league"]
player_match = player_match.sort_values(PLAYER_MATCH_SORT, ignore_index=True)

print(f"Player-match rows: {len(player_match):,}")
print(f"Unique players:    {player_match['playerId'].nunique():,}")
print(f"Unique matches:    {player_match['matchId'].nunique():,}")

## Step 4: Compute rolling last-5-match form features

For each player, shift match-level stats by 1 (so the current match is excluded),
then compute rolling sums over the previous 5 matches played.

**Leakage guard:** `shift(1)` ensures the current match never contributes to its
own form window. The first match for every player has all form features = 0.

**Important:** `player_match` must already be sorted by `PLAYER_MATCH_SORT` before
this block. The rolling windows depend on the ordering within each player group.
The re-sort below makes this dependency executable, not just documented.

In [ ]:
player_match = player_match.sort_values(PLAYER_MATCH_SORT, ignore_index=True)

g = player_match.groupby("playerId", group_keys=False)

# --- Core required form features ---

player_match["goals_last_5_matches"] = (
    g["match_goals"]
    .apply(lambda s: s.shift(1).rolling(5, min_periods=1).sum())
    .fillna(0)
    .astype("int32")
)
player_match["shots_last_5_matches"] = (
    g["match_shots"]
    .apply(lambda s: s.shift(1).rolling(5, min_periods=1).sum())
    .fillna(0)
    .astype("int32")
)
# shift(1) makes the first match NaN; rolling(min_periods=1).count() on an all-NaN
# window returns NaN, which fillna(0) converts to 0 — intentionally zero, not a bug.
player_match["matches_in_form_window"] = (
    g["match_shots"]
    .apply(lambda s: s.shift(1).rolling(5, min_periods=1).count())
    .fillna(0)
    .astype("int8")
)
player_match["conversion_rate_last_5_matches"] = np.where(
    player_match["shots_last_5_matches"] > 0,
    player_match["goals_last_5_matches"] / player_match["shots_last_5_matches"],
    0.0,
).astype("float32")

# --- Optional extra form features ---
# xg_pred encodes shot characteristics only (location, body part, etc.),
# not the outcome, so rolling prior-match xg_pred is not leakage.

player_match["xg_sum_last_5_matches"] = (
    g["match_xg_sum"]
    .apply(lambda s: s.shift(1).rolling(5, min_periods=1).sum())
    .fillna(0)
    .astype("float32")
)
player_match["goals_minus_xg_last_5_matches"] = (
    player_match["goals_last_5_matches"] - player_match["xg_sum_last_5_matches"]
).astype("float32")
player_match["shots_per_match_last_5_matches"] = np.where(
    player_match["matches_in_form_window"] > 0,
    player_match["shots_last_5_matches"] / player_match["matches_in_form_window"],
    0.0,
).astype("float32")
player_match["xg_per_shot_last_5_matches"] = np.where(
    player_match["shots_last_5_matches"] > 0,
    player_match["xg_sum_last_5_matches"] / player_match["shots_last_5_matches"],
    0.0,
).astype("float32")

print("Form features computed")
print(player_match[[
    "playerId", "matchId", "match_goals", "match_shots",
    "goals_last_5_matches", "shots_last_5_matches", "matches_in_form_window",
]].head())

## Step 5: Validate player-match rolling features

In [ ]:
first_matches = player_match.groupby("playerId", sort=False).head(1)
assert (first_matches["goals_last_5_matches"] == 0).all(), \
    "First player-match goals_last_5 not zero"
assert (first_matches["shots_last_5_matches"] == 0).all(), \
    "First player-match shots_last_5 not zero"
assert (first_matches["matches_in_form_window"] == 0).all(), \
    "First player-match matches_in_form_window not zero"

assert player_match["match_xg_sum"].ge(0).all(), "match_xg_sum contains negative values"

assert player_match["matches_in_form_window"].between(0, 5).all(), \
    "matches_in_form_window outside [0, 5]"
assert player_match["goals_last_5_matches"].ge(0).all()
assert player_match["shots_last_5_matches"].ge(0).all()
assert (player_match["goals_last_5_matches"] <= player_match["shots_last_5_matches"]).all(), \
    "goals_last_5 exceeds shots_last_5"
assert player_match["conversion_rate_last_5_matches"].between(0, 1).all(), \
    "conversion_rate_last_5 outside [0, 1]"

print("Player-match validation passed")

# --- Numerical leakage spot-check ---
# Verify shift(1) logic: row 0 should have all form = 0, row 1 should reflect row 0's stats.
eligible = (
    player_match[player_match["match_goals"] > 0]
    .groupby("playerId")
    .filter(lambda g: len(g) >= 2)
)
assert len(eligible) > 0, "No eligible player for spot-check"
sample_pid = eligible["playerId"].iloc[0]
pm_sample = player_match[player_match["playerId"] == sample_pid].head(5)
print(f"\nSpot-check player {sample_pid}:")
display(pm_sample[[
    "matchId", "match_datetime_utc", "match_goals", "match_shots",
    "goals_last_5_matches", "shots_last_5_matches", "matches_in_form_window",
]])

## Step 6: Merge form features back to shot level

Merge keys: `playerId` + `matchId`.
All shots by the same player in the same match receive identical form features.

In [ ]:
pre_merge_n = len(shots)
shots = shots.merge(
    player_match[ALL_FORM_COLS],
    on=["playerId", "matchId"],
    how="left",
    validate="many_to_one",
)

assert len(shots) == pre_merge_n, "Row count changed after form merge"

for col in [
    "goals_last_5_matches", "shots_last_5_matches",
    "conversion_rate_last_5_matches", "matches_in_form_window",
    "xg_sum_last_5_matches", "goals_minus_xg_last_5_matches",
    "shots_per_match_last_5_matches", "xg_per_shot_last_5_matches",
]:
    assert shots[col].notna().all(), f"Nulls in {col} after merge"

print(f"Post-merge rows: {len(shots):,}  (unchanged: {len(shots) == pre_merge_n})")

## Step 7: Shot-level QA — leakage safety

Verify that all shots by the same player in the same match share identical form values.
If any form feature varies within a player-match group, form is updating within the match
— which would be both incorrect and a leakage risk.

In [9]:
FORM_CHECK_COLS = [
    "goals_last_5_matches", "shots_last_5_matches",
    "conversion_rate_last_5_matches", "matches_in_form_window",
    "xg_sum_last_5_matches", "goals_minus_xg_last_5_matches",
    "shots_per_match_last_5_matches", "xg_per_shot_last_5_matches",
]

match_level_uniqueness = (
    shots.groupby(["playerId", "matchId"])[FORM_CHECK_COLS].nunique()
)
assert (match_level_uniqueness <= 1).all().all(), \
    "Form values vary within the same player-match — leakage detected"

print("Leakage QA passed: all shots in a player-match share identical form features")

Leakage QA passed: all shots in a player-match share identical form features


## Step 8: Coverage summaries

In [10]:
coverage = shots.groupby("dataset_split").agg(
    n_shots=("is_goal", "count"),
    share_with_form_history=("matches_in_form_window", lambda s: (s > 0).mean()),
    mean_matches_in_window=("matches_in_form_window", "mean"),
    mean_goals_last_5=("goals_last_5_matches", "mean"),
    mean_shots_last_5=("shots_last_5_matches", "mean"),
    mean_conversion_last_5=("conversion_rate_last_5_matches", "mean"),
).round(4)
print("Coverage by split:")
display(coverage)

top_players = (
    shots.groupby("playerId")
    .agg(
        n_shots=("is_goal", "count"),
        max_shots_last_5=("shots_last_5_matches", "max"),
        max_goals_last_5=("goals_last_5_matches", "max"),
    )
    .sort_values("max_shots_last_5", ascending=False)
    .head(10)
)
print("\nTop 10 players by max shots_last_5_matches:")
display(top_players)

Coverage by split:


,n_shots,share_with_form_history,mean_matches_in_window,mean_goals_last_5,mean_shots_last_5,mean_conversion_last_5
dataset_split,,,,,,
test,8881,0.9228,3.9474,1.0522,8.4324,0.1064
train,34159,0.9227,3.9294,1.0320,8.3487,0.1052



Top 10 players by max shots_last_5_matches:


,n_shots,max_shots_last_5,max_goals_last_5
playerId,,,
21385,172,39,3
3359,193,38,9
3322,170,37,11
8327,144,33,7
8717,175,33,8
40810,87,33,9
21384,110,30,10
14817,118,29,8
118,105,27,6


## Step 9: Split back into train and test

In [11]:
train_form = shots[shots["dataset_split"] == "train"].copy()
test_form  = shots[shots["dataset_split"] == "test"].copy()

assert len(train_form) == n_train, "Train row count changed after form computation"
assert len(test_form)  == n_test,  "Test row count changed after form computation"
assert set(train_form["league"].unique()) == {"France", "Germany", "Italy", "Spain"}, \
    "Wrong leagues in train_form"
assert set(test_form["league"].unique()) == {"England"}, \
    "Wrong leagues in test_form"

print(f"Train: {len(train_form):,}  |  Test: {len(test_form):,}")

Train: 34,159  |  Test: 8,881


## Step 10: Save form-enriched datasets

In [ ]:
NEW_FORM_COLS = [
    "goals_last_5_matches",
    "shots_last_5_matches",
    "conversion_rate_last_5_matches",
    "matches_in_form_window",
    "xg_sum_last_5_matches",
    "goals_minus_xg_last_5_matches",
    "shots_per_match_last_5_matches",
    "xg_per_shot_last_5_matches",
]

SAVE_COLS = [c for c in shots.columns if c != "dataset_split"]

train_form[SAVE_COLS].to_parquet(DATA_DIR / "wyscout_train_xg_form.parquet", index=False)
test_form[SAVE_COLS].to_parquet(DATA_DIR  / "wyscout_test_xg_form.parquet",  index=False)
shots[SAVE_COLS].head(100).to_csv(DATA_DIR / "wyscout_xg_form_sample.csv", index=False)

shots[SAVE_COLS].to_parquet(DATA_DIR / "wyscout_xg_form_combined.parquet", index=False)
player_match.to_csv(DATA_DIR / "wyscout_player_match_form_table.csv", index=False)

print(f"Saved {len(train_form):,} train → wyscout_train_xg_form.parquet")
print(f"Saved {len(test_form):,} test  → wyscout_test_xg_form.parquet")
print("Saved 100-row sample         → wyscout_xg_form_sample.csv")
print(f"\nNew form columns ({len(NEW_FORM_COLS)}): {NEW_FORM_COLS}")
print(f"Total saved columns: {len(SAVE_COLS)}")

## Step 11: Reload verification

In [ ]:
from src.datasets import verify_saved_xg_bundle

# Core four are contractually required
REQUIRED_FORM_COLS = {
    "goals_last_5_matches",
    "shots_last_5_matches",
    "conversion_rate_last_5_matches",
    "matches_in_form_window",
}
# Extra four are saved and verified
EXTRA_FORM_COLS = {
    "xg_sum_last_5_matches",
    "goals_minus_xg_last_5_matches",
    "shots_per_match_last_5_matches",
    "xg_per_shot_last_5_matches",
}

ALL_FORM_COLS = sorted(REQUIRED_FORM_COLS | EXTRA_FORM_COLS)

train_check, test_check = verify_saved_xg_bundle(
    "form", DATA_DIR,
    expected_train_rows=n_train,
    expected_test_rows=n_test,
    required_cols=ALL_FORM_COLS,
)

# ── Notebook-specific checks (dtypes, ranges) ──

for df_name, df in [("train", train_check), ("test", test_check)]:
    # Core form column dtypes
    assert pd.api.types.is_integer_dtype(df["goals_last_5_matches"]), \
        f"goals_last_5_matches not integer in {df_name}"
    assert pd.api.types.is_integer_dtype(df["shots_last_5_matches"]), \
        f"shots_last_5_matches not integer in {df_name}"
    assert pd.api.types.is_numeric_dtype(df["conversion_rate_last_5_matches"]), \
        f"conversion_rate_last_5_matches not numeric in {df_name}"
    assert pd.api.types.is_integer_dtype(df["matches_in_form_window"]), \
        f"matches_in_form_window not integer in {df_name}"

    # Core form column ranges
    assert df["matches_in_form_window"].between(0, 5).all()
    assert df["goals_last_5_matches"].ge(0).all()
    assert df["shots_last_5_matches"].ge(0).all()
    assert (df["goals_last_5_matches"] <= df["shots_last_5_matches"]).all()
    assert df["conversion_rate_last_5_matches"].between(0, 1).all()

    # Extra form column dtypes and ranges
    assert pd.api.types.is_numeric_dtype(df["xg_sum_last_5_matches"])
    assert df["xg_sum_last_5_matches"].ge(0).all()
    assert pd.api.types.is_numeric_dtype(df["shots_per_match_last_5_matches"])
    assert df["shots_per_match_last_5_matches"].ge(0).all()
    assert pd.api.types.is_numeric_dtype(df["xg_per_shot_last_5_matches"])
    assert df["xg_per_shot_last_5_matches"].between(0, 1).all()
    assert pd.api.types.is_numeric_dtype(df["goals_minus_xg_last_5_matches"])

print("Reload verification passed")
print(f"Train: {train_check.shape}  |  Test: {test_check.shape}")

display(train_check[[
    "playerId", "matchId", "is_goal",
    "career_shots_before_shot", "career_goals_before_shot", "career_conversion_rate_before_shot",
    "goals_last_5_matches", "shots_last_5_matches",
    "conversion_rate_last_5_matches", "matches_in_form_window",
]].head())